# Compare models
Same data/splits/seed for mlp, cnn, lstm, transformer; report val loss and test accuracy. Differences of ~1pp are noise at this sample size - check with walk-forward before believing any ranking.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import yaml, numpy as np, pandas as pd
from src.data.preprocess import load_ohlcv
dc = yaml.safe_load(open('../configs/data.yaml')); dc['path'] = '../' + dc['path']
bars = load_ohlcv(dc['path'], dc['resample']); bars.tail()

In [ ]:
from src.data.dataloader import build_loaders
from src.models.factory import build_model, MODELS
from src.training.trainer import Trainer, predict_loader
from src.evaluation.metrics import directional_accuracy
from src.utils.seed import set_seed
import torch
mc = yaml.safe_load(open('../configs/model.yaml')); tc = yaml.safe_load(open('../configs/training.yaml')); tc['epochs'] = 10
D = build_loaders(dc, bars); rows = []
for name in MODELS:
    set_seed(42); m = build_model({**mc, 'name': name}, D['n_features'], dc['seq_len']); h = Trainer(m, tc, torch.device('cpu')).fit(D['train'], D['val'])
    p, y = predict_loader(m, D['test'], torch.device('cpu')); rows.append((name, min(v for _, v in h), directional_accuracy(p, y)))
pd.DataFrame(rows, columns=['model', 'best_val_loss', 'test_acc'])